In [2]:
import torch
import sigpde.torch as sigpde
import math
import timeit
import csv

device = torch.device('cuda')

In [3]:
def brownian_motion(batch_size, length, dimension, device = torch.device('cpu'), dtype=torch.double):
  random_walks = torch.randn(batch_size, length, dimension, dtype = dtype, device = device) / math.sqrt(length)
  start = torch.zeros([batch_size, 1, dimension], device=device, dtype=dtype)
  random_walks = torch.cat((start, random_walks), dim=1)
  random_walks = torch.cumsum(random_walks, dim=1)
  return random_walks

def gaussian_noise(batch_size, length, dimension, device = torch.device('cpu'), dtype=torch.double):
  random_walks = torch.randn(batch_size, length, dimension, dtype = dtype, device = device) / math.sqrt(length)
  start = torch.zeros([batch_size, 1, dimension], device=device, dtype=dtype)
  random_walks = torch.cat((start, random_walks), dim=1)
  return random_walks

def add_time(x, start=0, stop=1):
    device = x.device
    dtype = x.dtype
    
    l = x.shape[1]

    t = torch.linspace(start, stop, l, device=device, dtype=dtype)
    t = t.unsqueeze(0).unsqueeze(-1)
    return torch.cat((x, t.expand(x.shape[0], x.shape[1], 1)), dim=-1)

In [5]:
def brownian_motion_add_time(batch_size, length, dimension, device=torch.device('cuda'), dtype=torch.double):
    x = brownian_motion(batch_size, length, dimension, device, dtype)
    return add_time(x)

def gaussian_noise_add_time(batch_size, length, dimension, device=torch.device('cuda'), dtype=torch.double):
    x = gaussian_noise(batch_size, length, dimension, device, dtype)
    return add_time(x)

In [7]:
def set_seed(seed):
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

In [35]:
# Warmup
static_kernel = sigpde.kernels.LinearKernel(1)
dyadic_order = 2
kernel = sigpde.RobustSigPDE(static_kernel, dyadic_order)
x = gaussian_noise_add_time(10, 200, 500, device, torch.double)

kernel.normalization(x, method="newton_raphson")
kernel.normalization(x, method="chandrupatla")
kernel.normalization(x, method="bisection")
kernel.normalization(x, method="secant")

tensor([0.1742, 0.1736, 0.1739, 0.1739, 0.1739, 0.1738, 0.1735, 0.1744, 0.1747,
        0.1738], device='cuda:0', dtype=torch.float64)

In [30]:
kernel.normalization(x, method="secant")

tensor([0.1740, 0.1735, 0.1741, 0.1743, 0.1729, 0.1740, 0.1736, 0.1736, 0.1731,
        0.1740], device='cuda:0', dtype=torch.float64)

In [36]:
kernel.normalization(x, method="newton_raphson")

tensor([0.1742, 0.1736, 0.1739, 0.1739, 0.1739, 0.1738, 0.1735, 0.1744, 0.1747,
        0.1738], device='cuda:0', dtype=torch.float64)

In [39]:
%timeit -n 10 -r 1 kernel.normalization(x, method="newton_raphson", tol=1e-6)
%timeit -n 10 -r 1 kernel.normalization(x, method="secant", tol=1e-6)

64.8 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 10 loops each)
91.5 ms ± 0 ns per loop (mean ± std. dev. of 1 run, 10 loops each)


In [26]:
static_kernel = sigpde.kernels.LinearKernel(1)
dyadic_order = 2
kernel = sigpde.RobustSigPDE(static_kernel, dyadic_order)

batch_size = 100
length = 100
tol = 1e-6

sims = {
    "brownian_motion": lambda: brownian_motion_add_time(batch_size, length, 10, device, torch.double),
    "gaussian_noise": lambda: gaussian_noise_add_time(batch_size, length, 500, device, torch.double),
    "gaussian_noise_extra": lambda: gaussian_noise_add_time(batch_size, 200, 500, device, torch.double)
}

set_seed(237428)

execs = 100
reps = 30

with open("normalization.csv", "w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["experiment", "method", "batch_size", "length", "dyadic_order", "run", "result"])
    for sim_name, sim_func in sims.items():
        
        x = sim_func()
        
        methods = {
            "newton_raphson": lambda: kernel.normalization(x, tol=tol, method="newton_raphson"),
            "chandrupatla": lambda: kernel.normalization(x, tol=tol, method="chandrupatla"),
            "secant": lambda: kernel.normalization(x, tol=tol, method="secant"),
            "bisection": lambda: kernel.normalization(x, tol=tol, method="bisection")
        }
        
        for method, func in methods.items():
            torch.cuda.empty_cache()
            # Use timeit with the function directly
            timing_results = timeit.repeat(func, number=execs, repeat=reps)
            
            # Save results to CSV
            for run, result in enumerate(timing_results, start=1):
                writer.writerow([sim_name, method, batch_size, length, dyadic_order, run, result / execs])